<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Реализация класса Invoice</h2>

----

### Вариант задания №10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о фактурах за поставленные товары или оказанные услуги. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.
Требования к базовому классу Invoice:

• Атрибуты: Номер фактуры (InvoiceNumber), Дата выдачи (IssueDate), Общая
сумма (TotalAmount).

• Методы:

o CalculateTotal(): метод для расчета общей суммы по фактуре.

o AddLine(LineItem lineItem): метод для добавления позиции в фактуру.

o RemoveLine(LineItem lineItem): метод для удаления позиции из фактуры.

Требования к производным классам:

1. ТоварнаяФактура (GoodsInvoice): Должна содержать дополнительные атрибуты, такие как Дата поставки (SupplyDate). Метод AddLine() должен быть переопределен для добавления информации о дате поставки товара при добавлении позиции.

2. УслуговаяФактура (ServiceInvoice): Должна содержать дополнительные атрибуты, такие как Дата оказания услуги (ServiceDate). Метод RemoveLine() должен быть переопределен для добавления информации о причине аннулирования услуги при удалении позиции.

3. КомбинированнаяФактура (CombinedInvoice) (если требуется третий класс): Должна содержать дополнительные атрибуты, такие как Наличие возврата (ReturnAllowed). Метод CalculateTotal() должен быть переопределен для учета возможного возврата товара или услуги при расчете общей суммы.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [16]:
using System;
using System.Collections.Generic;
using System.Linq;

public class LineItem
{
    public string Description { get; set; }
    public int Quantity { get; set; }
    public decimal UnitPrice { get; set; }
    public decimal Total => Quantity * UnitPrice;

    public LineItem(string description, int quantity, decimal unitPrice)
    {
        Description = description;
        Quantity = quantity;
        UnitPrice = unitPrice;
    }

    public override string ToString() => $"{Description} x{Quantity} = {Total:F2} руб.";
}

public abstract class Invoice
{
    public string InvoiceNumber { get; set; }
    public DateTime IssueDate { get; set; }
    public decimal TotalAmount { get; protected set; }
    public string CustomerName { get; set; }
    public string Currency { get; set; } = "RUB";
    public decimal TaxRate { get; set; } = 0.20m;
    public string Description { get; set; }

    protected List<LineItem> items = new List<LineItem>();

    public Invoice(string invoiceNumber, DateTime issueDate, string customerName)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
        CustomerName = customerName;
    }

    public Invoice(string invoiceNumber, DateTime issueDate, string customerName, string currency, decimal taxRate)
        : this(invoiceNumber, issueDate, customerName)
    {
        Currency = currency;
        TaxRate = taxRate;
    }

    public virtual decimal CalculateTotal()
    {
        decimal subtotal = items.Sum(i => i.Total);
        decimal tax = subtotal * TaxRate;
        TotalAmount = subtotal + tax;
        return TotalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        items.Add(lineItem);
        Console.WriteLine($"Добавлена позиция: {lineItem.Description}");
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (items.Remove(lineItem))
            Console.WriteLine($"Удалена позиция: {lineItem.Description}");
        else
            Console.WriteLine($"Позиция не найдена: {lineItem.Description}");
    }

    public virtual string GetInvoiceInfo()
    {
        return $"Фактура №{InvoiceNumber} от {IssueDate:dd.MM.yyyy}, клиент: {CustomerName}, итого: {TotalAmount:F2} {Currency}";
    }

    public void ApplyDiscount(decimal fixedDiscount)
    {
        TotalAmount -= fixedDiscount;
        Console.WriteLine($"Применена скидка {fixedDiscount:F2} {Currency}. Новая сумма: {TotalAmount:F2}");
    }

    public void ApplyDiscount(double percent)
    {
        decimal discount = TotalAmount * (decimal)(percent / 100);
        TotalAmount -= discount;
        Console.WriteLine($"Применена скидка {percent}%. Новая сумма: {TotalAmount:F2}");
    }

    public virtual bool Validate()
    {
        return !string.IsNullOrWhiteSpace(InvoiceNumber) && IssueDate <= DateTime.Now && items.Any();
    }
}

public class GoodsInvoice : Invoice
{
    public DateTime SupplyDate { get; set; }
    public string WarehouseLocation { get; set; }
    public int NumberOfPackages { get; set; }
    public decimal ShippingCost { get; set; }
    public string Carrier { get; set; }

    public GoodsInvoice(string invoiceNumber, DateTime issueDate, string customerName, DateTime supplyDate)
        : base(invoiceNumber, issueDate, customerName)
    {
        SupplyDate = supplyDate;
    }

    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine($"   (Товар будет поставлен {SupplyDate:dd.MM.yyyy})");
    }

    public void AddLine(LineItem lineItem, DateTime customSupplyDate)
    {
        items.Add(lineItem);
        Console.WriteLine($"Добавлена позиция: {lineItem.Description} с индивидуальной датой поставки {customSupplyDate:dd.MM.yyyy}");
    }

    public override decimal CalculateTotal()
    {
        decimal subtotal = items.Sum(i => i.Total);
        decimal tax = subtotal * TaxRate;
        TotalAmount = subtotal + tax + ShippingCost;
        return TotalAmount;
    }

    public override string GetInvoiceInfo()
    {
        return base.GetInvoiceInfo() + $", дата поставки: {SupplyDate:dd.MM.yyyy}, кол-во упаковок: {NumberOfPackages}";
    }

    public override bool Validate()
    {
        return base.Validate() && SupplyDate >= IssueDate;
    }
}

public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }
    public int ServiceHours { get; set; }
    public decimal HourlyRate { get; set; }
    public string ServiceManager { get; set; }
    public bool IsCompleted { get; set; }
    public string CancellationReason { get; private set; }

    public ServiceInvoice(string invoiceNumber, DateTime issueDate, string customerName, DateTime serviceDate)
        : base(invoiceNumber, issueDate, customerName)
    {
        ServiceDate = serviceDate;
    }

    public override void RemoveLine(LineItem lineItem)
    {
        CancellationReason = "Аннулировано по умолчанию (причина не указана)";
        base.RemoveLine(lineItem);
        Console.WriteLine($"Причина аннулирования: {CancellationReason}");
    }

    public void RemoveLine(LineItem lineItem, string reason)
    {
        CancellationReason = reason;
        if (items.Remove(lineItem))
            Console.WriteLine($"Удалена позиция: {lineItem.Description}. Причина: {reason}");
        else
            Console.WriteLine($"Позиция не найдена: {lineItem.Description}");
    }

    public override decimal CalculateTotal()
    {
        decimal subtotal;
        if (items.Any())
            subtotal = items.Sum(i => i.Total);
        else
            subtotal = ServiceHours * HourlyRate;

        decimal tax = subtotal * TaxRate;
        TotalAmount = subtotal + tax;
        return TotalAmount;
    }

    public override string GetInvoiceInfo()
    {
        return base.GetInvoiceInfo() + $", дата услуги: {ServiceDate:dd.MM.yyyy}, менеджер: {ServiceManager}";
    }
}

public class CombinedInvoice : Invoice
{
    public bool ReturnAllowed { get; set; }
    public decimal ReturnDiscount { get; set; }
    public DateTime ReturnDeadline { get; set; }
    public string ReturnReason { get; set; }

    public CombinedInvoice(string invoiceNumber, DateTime issueDate, string customerName, bool returnAllowed)
        : base(invoiceNumber, issueDate, customerName)
    {
        ReturnAllowed = returnAllowed;
    }

    public override decimal CalculateTotal()
    {
        decimal subtotal = items.Sum(i => i.Total);
        decimal tax = subtotal * TaxRate;
        TotalAmount = subtotal + tax;

        if (ReturnAllowed && DateTime.Now <= ReturnDeadline)
        {
            TotalAmount -= ReturnDiscount;
            Console.WriteLine($"Учтён возврат: скидка {ReturnDiscount:F2} {Currency} до {ReturnDeadline:dd.MM.yyyy}");
        }
        return TotalAmount;
    }

    public void AddLine(LineItem lineItem, string type)
    {
        items.Add(lineItem);
        Console.WriteLine($"Добавлена {type}: {lineItem.Description}");
    }

    public override string GetInvoiceInfo()
    {
        return base.GetInvoiceInfo() + $", возврат разрешён: {ReturnAllowed}, срок возврата: {ReturnDeadline:dd.MM.yyyy}";
    }
}

public class InvoiceRepository<T> where T : Invoice
{
    private List<T> invoices = new List<T>();

    public void Add(T invoice)
    {
        invoices.Add(invoice);
        Console.WriteLine($"Фактура {invoice.InvoiceNumber} добавлена в репозиторий.");
    }

    public bool Remove(string invoiceNumber)
    {
        var invoice = invoices.FirstOrDefault(i => i.InvoiceNumber == invoiceNumber);
        if (invoice != null)
        {
            invoices.Remove(invoice);
            Console.WriteLine($"Фактура {invoiceNumber} удалена из репозитория.");
            return true;
        }
        Console.WriteLine($"Фактура {invoiceNumber} не найдена.");
        return false;
    }

    public T FindByNumber(string invoiceNumber)
    {
        return invoices.FirstOrDefault(i => i.InvoiceNumber == invoiceNumber);
    }

    public void PrintAll()
    {
        Console.WriteLine("\n=== Все фактуры в репозитории ===");
        foreach (var inv in invoices)
        {
            Console.WriteLine(inv.GetInvoiceInfo());
        }
    }
}

class Program
{
    static void Main()
    {
        Console.OutputEncoding = System.Text.Encoding.UTF8;
        Console.WriteLine("=== Демонстрация системы фактур ===\n");

        GoodsInvoice goodsInv = new GoodsInvoice("G-001", DateTime.Now.AddDays(-5), "ООО Ромашка", DateTime.Now.AddDays(3))
        {
            WarehouseLocation = "Склад №5",
            NumberOfPackages = 10,
            ShippingCost = 500m,
            Carrier = "ПЭК",
            TaxRate = 0.20m,
            Currency = "RUB",
            Description = "Поставка электроники"
        };

        ServiceInvoice serviceInv = new ServiceInvoice("S-002", DateTime.Now.AddDays(-2), "ИП Иванов", DateTime.Now.AddDays(1))
        {
            ServiceHours = 40,
            HourlyRate = 1500m,
            ServiceManager = "Петрова А.А.",
            IsCompleted = false,
            TaxRate = 0.20m
        };

        CombinedInvoice combinedInv = new CombinedInvoice("C-003", DateTime.Now, "ЗАО ТехноСервис", true)
        {
            ReturnDiscount = 2000m,
            ReturnDeadline = DateTime.Now.AddDays(14),
            ReturnReason = "Гарантийный возврат",
            TaxRate = 0.20m
        };

        LineItem item1 = new LineItem("Ноутбук", 2, 45000m);
        LineItem item2 = new LineItem("Мышь", 5, 800m);
        goodsInv.AddLine(item1);
        goodsInv.AddLine(item2);
        goodsInv.AddLine(new LineItem("Клавиатура", 1, 3500m), DateTime.Now.AddDays(5));

        LineItem serviceItem = new LineItem("Консультация юриста", 1, 5000m);
        serviceInv.AddLine(serviceItem);

        LineItem combItem1 = new LineItem("Принтер", 1, 12000m);
        LineItem combItem2 = new LineItem("Настройка ПО", 1, 3000m);
        combinedInv.AddLine(combItem1);
        combinedInv.AddLine(combItem2);
        combinedInv.AddLine(new LineItem("Доставка", 1, 1500m), "услуга");

        Console.WriteLine("\n=== Расчёт сумм ===");
        Console.WriteLine($"Итого по товарной фактуре: {goodsInv.CalculateTotal():F2} руб. (включая доставку {goodsInv.ShippingCost:F2} руб.)");
        Console.WriteLine($"Итого по услуговой фактуре: {serviceInv.CalculateTotal():F2} руб.");
        Console.WriteLine($"Итого по комбинированной фактуре: {combinedInv.CalculateTotal():F2} руб.");

        Console.WriteLine("\n=== Удаление позиции из услуговой фактуры ===");
        serviceInv.RemoveLine(serviceItem);
        LineItem tempItem = new LineItem("Временная услуга", 1, 1000m);
        serviceInv.AddLine(tempItem);
        serviceInv.RemoveLine(tempItem, "Клиент передумал");

        Console.WriteLine("\n=== Применение скидок ===");
        goodsInv.ApplyDiscount(1000m);
        goodsInv.ApplyDiscount(10.0);

        Console.WriteLine("\n=== Работа с generic репозиторием ===");
        InvoiceRepository<Invoice> repository = new InvoiceRepository<Invoice>();
        repository.Add(goodsInv);
        repository.Add(serviceInv);
        repository.Add(combinedInv);
        repository.PrintAll();

        var found = repository.FindByNumber("S-002");
        if (found != null)
            Console.WriteLine($"\nНайдена фактура: {found.GetInvoiceInfo()}");

        Console.WriteLine("\n=== Проверка валидности ===");
        Console.WriteLine($"Товарная фактура валидна? {goodsInv.Validate()}");
        Console.WriteLine($"Услуговая фактура валидна? {serviceInv.Validate()}");
        Console.WriteLine($"Комбинированная фактура валидна? {combinedInv.Validate()}");

        // Ожидание нажатия любой клавиши без вывода сообщения (чтобы консоль не закрылась сразу)
        Console.ReadKey(true);
    }
}